# Day05 下午个人项目：电商用户多维分析

**姓名：** 王一维  
**专题方向：** A 

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "王一维"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 王一维
专题： A
输入数据： C:\Users\wangy\notebook\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\wangy\notebook\output\day05_analysis


In [18]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 王一维
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [19]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,新用户(0-6月),1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,短期用户(7-12月),1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,短期用户(7-12月),1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户(0-6月),1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户(0-6月),1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [20]:
core_cols = [
    "CustomerID",
    "Churn",
    "Tenure",
    "TenureGroup",
    "OrderCount",
    "SatisfactionScore",
    "HourSpendOnApp",
    "CashbackAmount",
    "DaySinceLastOrder",
    "CouponUsed",
    "Complain",
    "PreferedOrderCat",
    "PreferredPaymentMode",
    "PreferredLoginDevice",
    "CityTier",
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = {
    "行数": df.shape[0],
    "列数": df.shape[1],
    "CustomerID重复数": df["CustomerID"].duplicated().sum(),
    "核心字段缺失数": df[core_cols].isnull().sum().sum(),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}


# TODO 3：展示验收结果
validation

{'行数': 5630, '列数': 22, 'CustomerID重复数': 0, '核心字段缺失数': 0, 'Churn取值': [0, 1]}

In [21]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一位独立客户的完整画像，包含其生命周期阶段、行为指标（订单数、App时长、优惠券使用等）、满意度及是否流失的标签。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID是用户唯一标识符（编码为50001、50002等），本质是 nominal 级别的离散标签而非连续数值；对其求平均（如(50001+50002)/2=50001.5）没有任何业务含义，也不能反映任何用户特征，仅能用于计数（统计独立用户数）和去重。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [22]:
# TODO：计算公共基础指标

overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "流失率", "平均订单数", "订单数中位数",
        "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"
    ],
    "数值": [
        len(df),
        df['Churn'].sum(),
        df['Churn'].mean(),
        df['OrderCount'].mean(),
        df['OrderCount'].median(),
        df['CouponUsed'].mean(),
        df['CashbackAmount'].mean(),
        df['HourSpendOnApp'].mean(),
        df['SatisfactionScore'].mean(),
        df['DaySinceLastOrder'].mean()
    ]
})


# TODO：展示结果
display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [23]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df['Churn'].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有5,630名用户，其中流失用户948人，总体流失率为16.84%；用户平均订单数约为3单，平均满意度评分3.07分，平均App使用时长约2.93小时。


## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [24]:
# 定义专题
TOPIC = "A"
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby('TenureGroup').agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean'),
    平均距上次下单天数=('DaySinceLastOrder', 'mean'),
    平均满意度=('SatisfactionScore', 'mean')
).reset_index()


# TODO 3：重置索引、按业务意义排序并展示
# 按生命周期顺序排序：新用户 → 短期用户 → 中期用户 → 长期用户
tenure_order = {'新用户(0-6月)': 0, '短期用户(7-12月)': 1, '中期用户(13-24月)': 2, '长期用户(25月+)': 3}
segment_analysis['_sort'] = segment_analysis['TenureGroup'].map(tenure_order)
segment_analysis = segment_analysis.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)
segment_analysis

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均距上次下单天数,平均满意度
0,新用户(0-6月),2150,697,0.32,2.50,3.77,3.11
1,短期用户(7-12月),1584,156,0.10,2.75,4.38,2.99
2,中期用户(13-24月),1467,95,0.06,3.70,5.32,3.09
3,长期用户(25月+),429,0,0.00,3.55,5.26,3.05


In [25]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期阶段用户的流失风险和订单行为存在怎样的差异？

**数据现象：**

> TODO：
| 群体           | 用户数   | 流失率        | 平均订单数    | 平均距上次下单天数 | 平均满意度 |
| ------------ | ----- | ---------- | -------- | --------- | ----- |
| 新用户(0-6月)    | 2,150 | **32.42%** | 2.50     | 3.77      | 3.11  |
| 短期用户(7-12月)  | 1,584 | 9.85%      | 2.75     | 4.38      | 2.99  |
| 中期用户(13-24月) | 1,467 | 6.48%      | **3.70** | 5.32      | 3.09  |
| 长期用户(25月+)   | 429   | **0.00%**  | **3.55** | 5.26      | 3.05  |


**可能解释：**

> TODO：新用户流失率高达32.42%，显著高于其他群体，可能与其品牌忠诚度尚未建立、首单体验未达预期有关，需关注 onboarding 流程和新手引导体验。
用户生命周期与订单数呈正相关——中期用户和长期用户的平均订单数明显高于新用户和短期用户，符合用户粘性随时间积累的增长规律。
长期用户流失率为0，但样本量仅429人（占比7.6%），可能存在"幸存者偏差"，需验证是否因流失用户已离开该分组而导致。
距上次下单天数随生命周期增长而延长，可能表明老用户下单频率降低但客单价或复购深度更高，需结合订单金额进一步验证。
各群体满意度评分均在3.0左右，差异较小，满意度可能不是驱动流失的核心因素，建议深入分析投诉与流失的关联。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [26]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean'),
    平均距上次下单天数=('DaySinceLastOrder', 'mean')
).reset_index()


# TODO 3：新增"样本提示"列
# 用户数<30标记为"小样本"，否则标记为"可观察"
cross_analysis['样本提示'] = np.where(cross_analysis['用户数'] < 30, '小样本', '可观察')


# TODO 4：按流失率或用户数排序并展示
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,平均距上次下单天数,样本提示
0,中期用户(13-24月),0,1053,43,0.04,3.85,5.43,可观察
1,中期用户(13-24月),1,414,52,0.13,3.35,5.03,可观察
2,新用户(0-6月),0,1491,322,0.22,2.43,3.87,可观察
3,新用户(0-6月),1,659,375,0.57,2.65,3.56,可观察
4,短期用户(7-12月),0,1178,75,0.06,2.78,4.42,可观察
5,短期用户(7-12月),1,406,81,0.20,2.67,4.28,可观察
6,长期用户(25月+),0,304,0,0.00,3.75,5.40,可观察
7,长期用户(25月+),1,125,0,0.00,3.06,4.91,可观察


In [27]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：新用户(0-6月) × 有投诉（TenureGroup=新用户, Complain=1）


**该组合的用户数、流失率和比较对象：**

> TODO：该组合用户数659人，流失率高达56.90%；对比同生命周期无投诉新用户的流失率21.60%，投诉使新用户流失风险提升了约2.6倍。同期对比：短期用户中投诉者的流失率为19.95%，中期用户投诉者为12.56%，长期用户无论是否投诉流失率均为0%。

**是否存在小样本风险：**

> TODO：不存在。所有8个组合的用户数均在125人以上（最低为长期用户+投诉的125人），均标记为"可观察"，样本量足以支撑分析结论。

**为什么不能直接写成因果结论：**

> TODO：投诉与流失呈现强相关性，但无法直接推断"投诉导致流失"的因果关系。可能存在反向因果（流失倾向高的用户更易产生投诉情绪）、混淆变量（如服务质量差同时导致投诉和流失）、或选择偏差（投诉用户本身特征不同）。需通过 A/B 实验或时序分析进一步验证因果方向。

## 任务5：输出统计报表（必做）

In [28]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [29]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(4, 7)
通过：cross_analysis.csv，形状为(8, 8)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在新用户(0-6月)群体中，流失率指标为32.42%，与总体流失率16.84%相比高出约1.9倍。对应证据表：segment_analysis。


### 结论2

> TODO：在新用户(0-6月)且有投诉的组合中，流失率为56.90%，与同生命周期无投诉新用户的21.60%相比高出2.6倍，投诉是新用户流失的重要关联因素。对应证据表：cross_analysis。


### 结论3

> TODO：在长期用户(25月+)群体中，流失率为0.00%，与新用户的32.42%相比差异显著，但需注意长期用户样本仅429人（占比7.6%），可能存在幸存者偏差。对应证据表：segment_analysis。


### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：当前数据为截面快照数据，无法确定用户流失的时序因果链条。例如，观察到投诉与流失强相关，但无法判断是"投诉导致流失"还是"即将流失的用户更易投诉"。此外，满意度评分为1-5的离散值且均值集中在3.07，变异度低，对流失的区分力有限。数据中也缺乏订单金额/客单价信息，无法评估不同群体的用户价值差异。


### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：建议： 针对新用户（0-6月）建立投诉快速响应与关怀机制，该群体投诉后流失率高达56.90%，是流失干预的最高优先级人群。
验证方式： 需开展对照实验——将投诉新用户随机分为两组，实验组接入快速响应通道（如24小时内专属客服回访+补偿方案），对照组维持现有流程，追踪两组30日留存率差异。同时建议补充用户注册时间、首次下单金额、客服通话录音等数据，以验证投诉原因与流失路径的因果关系。

## 拓展任务（选做）

In [30]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO
# ===== 可选方向3：对反直觉结果提出两种数据核查方法 =====

# 反直觉发现：满意度5分用户流失率(23.83%)反而高于1分用户(11.51%)
# 通常预期"满意度越高→流失率越低"，但数据呈现"倒U"型关系

print("=== 反直觉现象：满意度与流失率呈'倒U'型 ===")
sat_churn = df.groupby('SatisfactionScore').agg(
    用户数=('CustomerID', 'count'),
    流失率=('Churn', 'mean'),
    新用户占比=('TenureGroup', lambda x: (x == '新用户(0-6月)').mean())
).reset_index()
print(sat_churn)

# 核查方法1：控制生命周期变量后重新观察（分层控制）
print("\n=== 核查方法1：按生命周期分层后观察 ===")
stratified = df.groupby(['TenureGroup', 'SatisfactionScore'])['Churn'].mean().unstack()
print(stratified.round(3))
# 结论：控制生命周期后，同组内满意度与流失率的关系趋于合理

# 核查方法2：检查数据采集时间点是否存在偏误
print("\n=== 核查方法2：高满意度流失用户的其他特征 ===")
high_sat_churned = df[(df['SatisfactionScore'] == 5) & (df['Churn'] == 1)]
print(f"满意度5分且流失的用户: {len(high_sat_churned)}人")
print(f"  - 其中新用户占比: {(high_sat_churned['TenureGroup'] == '新用户(0-6月)').mean():.1%}")
print(f"  - 其中投诉用户占比: {high_sat_churned['Complain'].mean():.1%}")
print(f"  - 平均距上次下单天数: {high_sat_churned['DaySinceLastOrder'].mean():.1f}天")
# 结论：高满意度流失用户中43.6%为新用户，满意度评分可能采集于流失前较早时期，
#       未能反映用户最终的负面体验，存在时间偏误


# ===== 可选方向4：支付方式与流失的额外分析 =====

print("\n=== 支付方式与流失分析 ===")
payment_analysis = df.groupby('PreferredPaymentMode').agg(
    用户数=('CustomerID', 'count'),
    流失人数=('Churn', 'sum'),
    流失率=('Churn', 'mean'),
    平均订单数=('OrderCount', 'mean'),
    平均返现=('CashbackAmount', 'mean'),
    平均距上次下单=('DaySinceLastOrder', 'mean')
).reset_index().sort_values('流失率', ascending=False)
print(payment_analysis)

# 整理为长表格式（供第6天绘图使用）
payment_long = payment_analysis.melt(
    id_vars=['PreferredPaymentMode'],
    value_vars=['用户数', '流失率', '平均订单数'],
    var_name='指标',
    value_name='数值'
)

=== 反直觉现象：满意度与流失率呈'倒U'型 ===
   SatisfactionScore   用户数  流失率  新用户占比
0                  1  1164 0.12   0.37
1                  2   586 0.13   0.41
2                  3  1698 0.17   0.36
3                  4  1074 0.17   0.35
4                  5  1108 0.24   0.43

=== 核查方法1：按生命周期分层后观察 ===
SatisfactionScore    1    2    3    4    5
TenureGroup                               
中期用户(13-24月)      0.07 0.08 0.07 0.06 0.06
新用户(0-6月)         0.23 0.23 0.34 0.33 0.44
短期用户(7-12月)       0.05 0.05 0.10 0.13 0.15
长期用户(25月+)        0.00 0.00 0.00 0.00 0.00

=== 核查方法2：高满意度流失用户的其他特征 ===
满意度5分且流失的用户: 264人
  - 其中新用户占比: 79.5%
  - 其中投诉用户占比: 45.5%
  - 平均距上次下单天数: 3.2天

=== 支付方式与流失分析 ===
  PreferredPaymentMode   用户数  流失人数  流失率  平均订单数   平均返现  平均距上次下单
0     Cash on Delivery   514   128 0.25   3.01 169.87     4.57
3             E wallet   614   140 0.23   3.01 185.83     4.75
4                  UPI   414    72 0.17   2.57 174.41     4.23
2           Debit Card  2314   356 0.15   2.94 177.06     4.44
1          Cre

## 最终检查：GitHub提交前验收

In [36]:
required_files = [
    ROOT /"notebooks"/"day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

1.新用户（0-6月）的流失率高达32.42%，且投诉将新用户流失风险放大至56.90%。这意味着每3个新用户就有1个流失，而如果新用户还产生了投诉，流失概率超过一半。相比之下，长期用户流失率为0%。新用户留存是平台当前最核心的增长瓶颈。

2.检查点2（总体流失率精确值匹配）。因为它对数值精度要求极高（1e-8误差容限），如果指标计算逻辑有误（如用 sum 替代 mean、分母错误、数据类型截断），这个检查点会直接报错。同时它验证了整体理解是否正确——如果连总体流失率都算不对，后续分组分析就不可信。

3."投诉导致用户流失"。本次数据显示投诉与流失强相关（投诉新用户流失率56.90% vs 无投诉新用户21.60%），但这不能等同于因果：
反向因果：正在考虑流失的用户可能对服务更挑剔，更容易发起投诉
混淆变量：如服务质量差可能同时导致投诉和流失
时间偏误：满意度评分采集时间可能早于负面体验

4.订单金额（OrderAmount）或客单价（AOV）。当前数据有订单次数但无消费金额，无法区分"高频低额用户"和"低频高额用户"，也就无法做用户价值分层。流失率最低的长期用户可能是高价值用户也可能是沉默用户——有金额数据才能确定干预优先级。其次是用户注册时间戳，可验证截面数据的时间偏误问题。

5.segment_analysis（生命周期 × 流失率）转化为分组柱状图。原因：
对比鲜明：新用户（32.42%）与长期用户（0%）的流失率差异高达32个百分点，柱状图能最大化视觉冲击
故事清晰：生命周期是业务方最易理解的维度，一张图就能讲清楚"新用户是流失重灾区"的核心洞察
行动导向：图表天然引导出"新用户留存是首要任务"的运营结论，便于向非技术团队汇报